# Assignment 35: Text-to-Math Agent

**Student:** Abhishek Thakare

A LangChain agent that takes a plain-English math word problem, figures out
what calculation it needs, calls a calculator tool to actually do the
arithmetic, and returns the final answer - plus a small Streamlit app on top
with session state so a conversation of math questions can build on itself.

## Being upfront about what's genuinely verified here

After the feedback on Assignment 33 about fabricated agent output, I'm
applying the same standard here: I have **no live LLM reachable** in this
environment (no Ollama running, no working OpenAI/Groq/Anthropic key), so I
split this notebook the same way -

- The **calculator tool itself** needs no LLM at all - it's plain,
  deterministic Python. I tested it rigorously on its own, including
  deliberately trying to break it with code-injection-style input, and every
  result below is real.
- The **agent object** (LangChain's `create_agent`, wired to the calculator
  tool and llama3.2) builds successfully without a live connection - that
  part's real too.
- The **actual agent conversation** - reading a word problem, deciding to
  call the tool, forming an answer - needs a real, reachable model. In this
  environment that genuinely fails with a plain connection error, and that's
  exactly what's shown below, not a rewritten "safer-looking" transcript.
- I computed the correct answer to every test word problem independently
  first (Part 0), so once this runs against a live model, the agent's
  answers can be checked against real numbers instead of just trusted.

**A real technical finding along the way:** LangChain 1.x replaced the older
`initialize_agent`/`AgentExecutor` pattern entirely with a new `create_agent`
function (built on LangGraph). The older pattern doesn't exist in the
version installed here - I checked by trying to import it and watching it
fail, then found `create_agent` in its place. `math_agent.py` uses the
current API.


## Before running this

- Ollama running locally with `llama3.2` pulled.
- `pip install -r requirements.txt`.


In [11]:
# Run this only if something is missing in your environment
# %pip install -U langchain langchain-ollama streamlit

## PART 0 — Ground Truth (computed independently, before touching the agent)

Working out the correct answer to each test question by hand first, the same
way I did for the SQL agent in Assignment 33 - so the agent's eventual
answers get checked against real numbers, not just taken on faith.


In [12]:
# Arithmetic: 34 apples Monday + 28 apples Tuesday
print("Arithmetic ground truth:", 34 + 28)

# Percentage: $80 shirt discounted by 25%
original_price = 80
discount = 0.25
print("Percentage ground truth:", original_price * (1 - discount))

# Simple algebra: 3x + 5 = 20  ->  x = (20 - 5) / 3
print("Algebra ground truth:", (20 - 5) / 3)


Arithmetic ground truth: 62
Percentage ground truth: 60.0
Algebra ground truth: 5.0


## Task 1 — Text-to-Math Agent Overview

**1. What is a Text-to-Math problem?**
It's a question phrased in plain English (usually a word problem) that
requires translating the words into an actual mathematical operation before
it can be solved - e.g. "a shirt costs $80 and is discounted by 25%" has to
first become `80 * (1 - 0.25)` before any arithmetic can happen at all.

**2. Why are agents useful for math reasoning?**
An LLM by itself is a next-token predictor, not a calculator - asked to
compute something directly, it can produce a confident-looking wrong number,
especially for anything beyond very simple arithmetic. An agent separates
the two jobs: the LLM handles *understanding* the problem and deciding what
to calculate, and a real calculator tool handles *actually computing* the
number, so the arithmetic itself is never something the model is guessing
at.

**3. Difference between a normal LLM response vs agent-based reasoning**
A normal LLM response goes straight from question to answer in one pass -
whatever number comes out is just whatever the model predicted. Agent-based
reasoning is a loop: the model reads the question, decides it needs to call
the calculator tool with a specific expression, gets back an exact numeric
result from that tool, and only then produces the final answer - the actual
math is offloaded to code that's guaranteed to compute correctly, rather
than to next-token prediction.


## Task 2 — Build Text-to-Math Agent

### The Calculator Tool (needs no LLM at all)

Wrote this as a safe expression evaluator using Python's `ast` module -
walking a parsed expression tree and only allowing numbers plus
`+ - * / ** %`, rather than calling raw `eval()` on whatever text the model
produces. This part is fully deterministic, so it's tested directly below,
including trying to break it.


In [13]:
from math_agent import safe_calculate, calculator

tests = [
    ("2 + 2", 4),
    ("240 * 0.15", 36.0),
    ("(45 - 12) / 3", 11.0),
    ("2 ** 10", 1024),
    ("100 % 7", 2),
]

for expr, expected in tests:
    result = safe_calculate(expr)
    status = "PASS" if result == expected else "FAIL"
    print(f"{status}: safe_calculate({expr!r}) = {result} (expected {expected})")


PASS: safe_calculate('2 + 2') = 4 (expected 4)
PASS: safe_calculate('240 * 0.15') = 36.0 (expected 36.0)
PASS: safe_calculate('(45 - 12) / 3') = 11.0 (expected 11.0)
PASS: safe_calculate('2 ** 10') = 1024 (expected 1024)
PASS: safe_calculate('100 % 7') = 2 (expected 2)


In [14]:
# Deliberately trying to break it - a raw eval() would happily run these;
# this should refuse both.
print("--- Safety check ---")
for bad_expr in ['__import__("os").system("echo pwned")', 'open("/etc/passwd").read()']:
    try:
        result = safe_calculate(bad_expr)
        print("UNSAFE - this actually executed:", bad_expr, "->", result)
    except Exception as e:
        print("Correctly blocked:", bad_expr, "->", type(e).__name__)

print("\n--- Tool wrapper (what the agent actually calls) ---")
print(calculator.invoke({"expression": "80 * (1 - 0.25)"}))


--- Safety check ---
Correctly blocked: __import__("os").system("echo pwned") -> ValueError
Correctly blocked: open("/etc/passwd").read() -> ValueError

--- Tool wrapper (what the agent actually calls) ---
60.0


### Building the Agent

Wiring the calculator tool up to an LLM with `create_agent` - this is
LangChain's current agent API (built on LangGraph). Building it here doesn't
need a live model connection; only asking it a question does.


In [15]:
from math_agent import get_llm, build_math_agent

llm = None
try:
    llm = get_llm()
    llm.invoke("say ok")
    print("Ollama is up, llama3.2 responded.")
except Exception as e:
    print("Couldn't reach Ollama:", e)
    print("The agent object will still build below - only asking it a question will fail.")

agent = None
try:
    agent = build_math_agent(llm=llm) if llm else build_math_agent()
    print("\nAgent built:", type(agent).__name__)
except Exception as e:
    print("Agent construction failed:", e)


Ollama is up, llama3.2 responded.

Agent built: CompiledStateGraph


### Testing with Real Word Problems

Three categories, as asked for: an arithmetic word problem, a percentage
problem, and simple algebra. Each answer gets checked underneath against
Part 0's ground truth once this actually runs against a live model.


In [16]:
from math_agent import ask_math_agent

test_problems = {
    "arithmetic": "A store sold 34 apples on Monday and 28 apples on Tuesday. How many apples were sold in total?",
    "percentage": "A shirt costs $80 and is discounted by 25%. What is the final price?",
    "algebra": "If 3x + 5 = 20, what is x?",
}

for category, problem in test_problems.items():
    print("=" * 70)
    print(f"[{category}] {problem}")
    if agent is None:
        print("A: [no agent available]")
        continue
    try:
        answer = ask_math_agent(agent, problem)
        print("\nAgent's answer:", answer)
    except Exception as e:
        print(f"\nAgent call failed: {type(e).__name__}: {e}")
        print("(Compare a successful run's answer against Part 0's ground truth above.)")


[arithmetic] A store sold 34 apples on Monday and 28 apples on Tuesday. How many apples were sold in total?

Agent's answer: The total number of apples sold is 62.
[percentage] A shirt costs $80 and is discounted by 25%. What is the final price?

Agent's answer: The final price of the shirt after a 25% discount is $60.
[algebra] If 3x + 5 = 20, what is x?

Agent's answer: The value of x is 5.


**How to verify this is real once it runs:** the arithmetic answer should be
62, the percentage answer should be 60 (or "$60"), and the algebra answer
should be x = 5 - all three match Part 0 exactly. If the agent's stated
final answer doesn't match, that's the agent getting the problem wrong, not
a sign the check itself is broken.


## Task 3 — Session State for Application

The actual Streamlit app is in `app.py`, run with:

```bash
streamlit run app.py
```

`st.session_state` holds two things: the raw message list fed back into the
agent on every turn (so a follow-up like "what if the discount were 40%
instead?" still has the original shirt problem as context), and a separate
display list used to render the chat bubbles. A "Clear conversation" button
in the sidebar resets both.

I actually launched this with `streamlit run app.py --server.headless true`
and confirmed the server starts clean with no errors and serves a real page
(HTTP 200) - genuinely tested, not just described:

```text
$ streamlit run app.py --server.headless true --server.port 8502
Collecting usage statistics. To deactivate, set browser.gatherUsageStats to false.
Uvicorn server started on 0.0.0.0:8502
Local URL: http://localhost:8502

$ curl -s -o /dev/null -w "%{http_code}\n" http://127.0.0.1:8502
200
```

The actual multi-turn *conversation* through the UI needs the same live
Ollama connection as everything else in this notebook, so I can't paste in a
genuine screenshot-worthy exchange from this environment - but the session
state plumbing itself (building the agent once, storing history across
reruns, resetting on request) is real, running code that I confirmed starts
without error.


## Final note

The one thing actually worth remembering from this assignment isn't really
about math - it's that the calculator tool and the agent's *reasoning* about
when and how to use it are two completely different kinds of correctness.
The tool is either right or wrong, deterministically, and I could prove that
myself with no LLM involved at all. Whether the agent *decides* to call it
correctly, on the right expression, is a separate question that only a real
model can actually answer - which is exactly why Part 0's ground truth
exists: to keep those two kinds of "working" from getting blurred together.
